In [ ]:
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PriyangkaNong/Self-Supervised-Learning/blob/main/node_prediction.ipynb)

In [ ]:
!pip install torch-geometric

# Import libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv

In [ ]:
# Check device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


In [ ]:
# Load the Cora dataset

graph_dataset = Planetoid(root="./data/Cora", name="Cora")
data = graph_dataset[0]
print(data)

Processing...


Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


Done!


In [ ]:
print("Number of nodes:", data.num_nodes)
print("Number of edges:", data.num_edges)
print("Number of node features:", data.num_node_features)
print("Number of classes:", graph_dataset.num_classes)
print("Training nodes:", data.train_mask.sum().item())
print("Validation nodes:", data.val_mask.sum().item())
print("Test nodes:", data.test_mask.sum().item())

Number of nodes: 2708
Number of edges: 10556
Number of node features: 1433
Number of classes: 7
Training nodes: 140
Validation nodes: 500
Test nodes: 1000


In [ ]:
#Build the GCN Encoder
class GCNEncoder(nn.Module):
  def __init__(self, input_dim, hidden_dim, output_dim):
    super(GCNEncoder, self).__init__()
    self.conv1 = GCNConv(input_dim, hidden_dim)
    self.conv2 = GCNConv(hidden_dim, output_dim)
  def forward(self, x, edge_index):
    x = self.conv1(x, edge_index)
    x = F.relu(x)
    x = self.conv2(x, edge_index)
    return x

In [ ]:
#Create the encoder
graph_encoder = GCNEncoder(data.num_node_features, 128, 128).to(device)

In [ ]:
#Testing the GCN
graph_embeddings = graph_encoder(data.x.to(device), data.edge_index.to(device))
print("Graph Embedding Shape:",graph_embeddings.shape)

Graph Embedding Shape: torch.Size([2708, 128])


In [ ]:
#Create the node classifier
class NodeClassifier(nn.Module):
  def __init__(self, input_dim, num_classes):
    super(NodeClassifier, self).__init__()
    self.fc = nn.Linear(input_dim, num_classes)
  def forward(self,x):
    return self.fc(x)
node_classifier = NodeClassifier(input_dim=128, num_classes=graph_dataset.num_classes).to(device)

In [ ]:
#create the loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    list(graph_encoder.parameters()) +
    list(node_classifier.parameters()),
    lr=0.01
)

In [ ]:
#Train the node classifier
num_epochs = 100

for epoch in range(num_epochs):

    graph_encoder.train()
    node_classifier.train()

    optimizer.zero_grad()

    embeddings = graph_encoder(
        data.x.to(device),
        data.edge_index.to(device)
    )

    outputs = node_classifier(embeddings)

    loss = criterion(
        outputs[data.train_mask.to(device)],
        data.y.to(device)[data.train_mask.to(device)]
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{num_epochs}], "
            f"Loss: {loss.item():.4f}"
        )

Epoch [10/100], Loss: 0.0008
Epoch [20/100], Loss: 0.0000
Epoch [30/100], Loss: 0.0000
Epoch [40/100], Loss: 0.0000
Epoch [50/100], Loss: 0.0000
Epoch [60/100], Loss: 0.0000
Epoch [70/100], Loss: 0.0000
Epoch [80/100], Loss: 0.0000
Epoch [90/100], Loss: 0.0000
Epoch [100/100], Loss: 0.0000


In [ ]:
#Evaluate the Node prediction
graph_encoder.eval()
node_classifier.eval()

with torch.no_grad():

    embeddings = graph_encoder(
        data.x.to(device),
        data.edge_index.to(device)
    )

    outputs = node_classifier(embeddings)

    predictions = outputs.argmax(dim=1)

    test_mask = data.test_mask.to(device)

    correct = (
        predictions[test_mask] ==
        data.y.to(device)[test_mask]
    ).sum().item()

    total = test_mask.sum().item()

    test_accuracy = correct / total

print(f"Node Prediction Test Accuracy: {test_accuracy * 100:.2f}%")

Node Prediction Test Accuracy: 76.70%
